In [1]:
import types
import math
import time

# QoL improvements

## Итераторы

Итераторы нужны для того, чтобы можно было удобно проходить по элементам некоей сущности. Обычно при проходе по элементам списка мы делаем так:

In [2]:
num_list = [1, 2, 3]
for i in num_list:
  print(i)

1
2
3


Но можно создать итератор - объект, который может вернуть нам очередной элемент или же кинуть исключение

In [3]:
itr = iter(num_list)

In [4]:
type(itr)

list_iterator

In [5]:
print(next(itr))
print(next(itr))
print(next(itr))
print(next(itr))

1
2
3


StopIteration: 

Мы можем делать и собственные итераторы

In [6]:
class SimpleIterator:
  def __init__(self, limit):
    self.limit = limit
    self.counter = 0

  def __next__(self):
    if self.counter < self.limit:
      self.counter += 1
      return 1
    else:
      raise StopIteration

In [7]:
s_iter1 = SimpleIterator(3)
print(next(s_iter1))
print(next(s_iter1))
print(next(s_iter1))
print(next(s_iter1))

1
1
1


StopIteration: 

Является ли range итератором?

In [8]:
r = range(0, 3)

In [9]:
for i in r:
  print(i)

0
1
2


In [10]:
next(r)

TypeError: 'range' object is not an iterator

In [11]:
next(iter(r))

0

## Генераторы

Генератор - функция, которая определяет правила для __next__  
При этом вместо того, чтобы возвращать значение с помощью __return__, мы их __yield__'им

In [12]:
def simple_generator(val):
  while val > 0:
    val -= 1
    return val

In [14]:
simple_generator(3)

2

In [15]:
simple_generator(3)

2

In [16]:
def simple_generator(val):
  while val > 0:
    val -= 1
    yield val

In [17]:
g = simple_generator(3)

In [18]:
type(g)

generator

In [19]:
print(next(g))
print(next(g))
print(next(g))
print(next(g))

2
1
0


StopIteration: 

Важно учитывать, что:
- генератор не хранит в себе все значения
- генератор как бы замораживается в процессе работы, ожидая очередную свою итерацию
- можно добавить сообщение в конце!

In [20]:
def simple_generator(val):
  while val > 0:
    val -= 1
    yield 1
  return "the end!"


g = simple_generator(3)
print(next(g))
print(next(g))
print(next(g))
print(next(g))

1
1
1


StopIteration: the end!

Все генераторы - итераторы, но не наоборот. Ну и генераторы просто упрощают создание сложных итераторов

In [21]:
def squares(start, stop):
    for i in range(start, stop):
        yield i * i

generator = squares(1, 3)

In [23]:
next(generator)

4

## Форматирование строк

Когда надо складывать несколько строк, часто можно просто обойтись простыми средствами

In [24]:
name = "Ann"
example = "my name is " + name
print(example)

my name is Ann


In [25]:
header = "my name is"
name = "Ann"
example = header + " " + name
print(example)

my name is Ann


In [26]:
"a" * 10

'aaaaaaaaaa'

Но иногда хочется сделать красиво

In [27]:
example2 = f"my name is {'Ann'}"
print(example2)

my name is Ann


In [28]:
example3 = f"my name is {name}"
print(example3)

my name is Ann


In [29]:
example4 = f"my name is {name.lower()}"
print(example4)

my name is ann


А еще можно заранее подготовить шаблон, а потом использовать

In [30]:
template = "my {0} is {1}, {1} is my {0}"
print(template.format("name", name))

my name is Ann, Ann is my name


In [31]:
template2 = "my {field} is {name}, {name} is my {field}"
print(template2.format(field="name", name=name))

my name is Ann, Ann is my name


Также можно делать форматирование с конверсией типов (а всегда ли она отработает?)

In [32]:
"I love %s and %s" % ("dogs", "cats")

'I love dogs and cats'

In [33]:
"I love %(first)s and %(second)s" % {"first": "dogs", "second": "cats"}

'I love dogs and cats'

In [34]:
"I am %d years old" % 20

'I am 20 years old'

In [35]:
"I am %s years old" % 20

'I am 20 years old'

In [36]:
"I am %d years old" % 'a'

TypeError: %d format: a real number is required, not str

А можно ли в таком случае пользоваться своим классом?

In [37]:
class Test():
  def __init__(self, val):
    self.val = val

In [38]:
"I am %d years old" % Test(10)

TypeError: %d format: a real number is required, not Test

In [39]:
class Test():
  def __init__(self, val):
    self.val = val

  def __int__(self):
    return self.val

In [40]:
"I am %d years old" % Test(10)

'I am 10 years old'

In [41]:
class Test():
  def __init__(self, val):
    self.val = val

  def __real__(self):
    return self.val

In [42]:
"I am %d years old" % Test(10)

TypeError: %d format: a real number is required, not Test

Можно и без всяких прекрас просто поджойнить массив

In [43]:
data = ["I", "am", "Ann"]
result = ""
for item in data:
  result += item + " "

print(result)

I am Ann 


In [45]:
" ".join(["I", "am", "Ann"])

'I.am.Ann'

А вообще со сложением строк надо быть осторожнее, особенно когда дело касается запросов в базу или генерации html

In [46]:
def get_data(condition):
  query = f"Select * From table Where id = {condition}"
  print(query)

get_data(1)
get_data("1 or True")

Select * From table Where id = 1
Select * From table Where id = 1 or True


In [47]:
def make_html(text):
  html = f"<p>{text}</p>"
  with open("test.html", "w") as f:
    f.write(html)

In [48]:
make_html("test")

In [49]:
make_html("<script>alert('1')</script>")

## Comprehension

Обычно для заполнения массивов мы просто используем циклы

In [50]:
res = []
for i in range(0, 10):
  res.append(i)

res

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [51]:
res2 = []
for item in res:
  res2.append(item*2)

res2

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18]

Но такую запись можно и сократить

In [ ]:
res = []
for i in range(0, 10):
  res.append(i)

In [52]:
res = [i for i in range(0, 10)]
res

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [53]:
[i**2 for i in range(0, 10)]

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

Можно делать и двойные циклы

In [54]:
res = []
for x in [1,2,3]:
  for y in [3,4,5]:
    if x != y:
      res.append([x,y])
res

[[1, 3], [1, 4], [1, 5], [2, 3], [2, 4], [2, 5], [3, 4], [3, 5]]

In [55]:
[[x, y] for x in [1,2,3] for y in [3,4,5] if x != y]

[[1, 3], [1, 4], [1, 5], [2, 3], [2, 4], [2, 5], [3, 4], [3, 5]]

In [56]:
[(x, y) for x in [1,2,3] for y in [3,4,5] if x != y]

[(1, 3), (1, 4), (1, 5), (2, 3), (2, 4), (2, 5), (3, 4), (3, 5)]

Можно и использовать функции

In [57]:
def multiplier(a,b):
  return a*b

[multiplier(x,y) for x in [1,2,3] for y in [4,5,6]]

[4, 5, 6, 8, 10, 12, 12, 15, 18]

In [58]:
data = [(i, str(i)) for i in range(4)]

In [59]:
print(data)

[(0, '0'), (1, '1'), (2, '2'), (3, '3')]


И да, так можно делать не только с массивами, но и со словарями

In [60]:
res = {}
for i in range(4):
  res[i] = str(i)

print(res)

{0: '0', 1: '1', 2: '2', 3: '3'}


In [61]:
{i : str(i) for i in range(4)}

{0: '0', 1: '1', 2: '2', 3: '3'}

## Map, filter, reduce

Как-то мы уже говорили про функциональное программирование. Иногда написание кода в таком стиле может сделать код куда проще и лаконичнее

In [ ]:
# map(function_to_apply, *iterables) -> generator

In [62]:
items = [1, 2, 3, 4, 5]
res = []
for item in items:
  res.append(item**2)
print(res)

[1, 4, 9, 16, 25]


In [63]:
items = [1, 2, 3, 4, 5]
res = list(map(lambda x: x**2, items))
print(res)

[1, 4, 9, 16, 25]


Сделаем вызов чуть менее страшным

In [ ]:
def sq(x):
  return x ** 2

items = [1, 2, 3, 4, 5]
res = list(map(sq, items))
print(res)

А если функция принимает несколько значений?

In [65]:
def pow(x, y):
  return x ** y

items = [1, 2, 3, 4, 5]
res = list(map(pow, items))
print(res)

TypeError: pow() missing 1 required positional argument: 'y'

In [66]:
def pow(x, y):
  return x ** y

items = [1, 2, 3, 4, 5]
res = list(map(pow, items, 1))
print(res)

TypeError: 'int' object is not iterable

In [67]:
def pow(x, y):
  return x ** y

items = [1, 2, 3, 4, 5]
res = list(map(pow, items, len(items) * [2]))
print(res)

[1, 4, 9, 16, 25]


In [ ]:
# filter(function_to_apply, iterable) -> iterable

In [68]:
number_list = range(-5, 5)
less_than_zero = list(filter(lambda x: x < 0, number_list))
print(less_than_zero)

[-5, -4, -3, -2, -1]


In [69]:
def less_z(x):
  return x < 0

less_than_zero = list(filter(less_z, number_list))
print(less_than_zero)

[-5, -4, -3, -2, -1]


In [70]:
scores = {"Alice": 85, "Bob": 42, "Cleo": 91}
passed = dict(filter(lambda item: item[1] >= 60, scores.items()))
print(passed)

{'Alice': 85, 'Cleo': 91}


In [ ]:
# reduce(function_to_apply, iterable[, initial])

In [71]:
product = 1
items = [1, 2, 3, 4]
for item in items:
  product = product * item
print(product)

24


In [72]:
from functools import reduce
product = reduce((lambda x, y: x * y), [1, 2, 3, 4])
print(product)

24


Попробуем указать initial

In [74]:
product = reduce((lambda x, y: x * y), [1, 2, 3, 4], 10)
print(product)

240


Чтобы проще разобраться в том, что произошло, сделаем неанонимную функцию

In [75]:
def m(a, b):
  print(a, b)
  return a * b

product = reduce(m, [1, 2, 3, 4], 10)
print(product)

10 1
10 2
20 3
60 4
240


## Произвольные аргументы

Иногда мы хотим принимать неограниченное количество аргументов; можно для такого использовать те же массивы

In [76]:
def multiplier(start, items):
  for item in items:
    start *= item
  return start

In [77]:
multiplier(1, [2,3])

6

In [78]:
multiplier(1, [])

1

In [79]:
multiplier(1, [2,3,4,5,6])

720

Но можно воспользоваться механизмом произвольных аргументов

In [80]:
def multiplier(start, *items):
  for item in items:
    start *= item
  return start

In [81]:
multiplier(1)

1

In [82]:
multiplier(1, 2, 3)

6

In [83]:
print(multiplier(5))
print(multiplier(0,2,3,4))
print(multiplier(5,2,3,4,10,1))

5
0
1200


После массива  можно указать аргументы, но обращаться к ним придется только по имени

In [84]:
def multiplier2(start, *args, end):
  res = start
  for arg in args:
    res *= arg
  return res + end

In [85]:
multiplier2(1,2,3,4)

TypeError: multiplier2() missing 1 required keyword-only argument: 'end'

In [86]:
multiplier2(1,2,3,end=4)

10

In [87]:
def multiplier(start, *args):
  print(type(args))
  for arg in args:
    start *= arg
  return start

multiplier(1,2,3)

<class 'tuple'>


6

Можно также задавать массив именованных аргументов

In [88]:
def summer(start, **kwargs):
  print(type(kwargs))
  return start + kwargs["end"]

print(summer("s", middle="middle", middle2="m2", end="end"))

<class 'dict'>
send


In [89]:
def summer(start, **kwargs):
  res = start
  for k in kwargs:
    res += kwargs[k]
  return res

print(summer("s", middle="middle", middle2="m2"))

smiddlem2


Ну и, конечно, можно и совместить это все

In [90]:
def test(start, *args, **kwargs):
  res = start
  for item in args:
    res += item
  for k in kwargs:
    res += "{0}={1}".format(k, kwargs[k])
  return res

test("a", "b", "c", final="xyz")

'abcfinal=xyz'

## Значения по умолчанию

Для аргументов можно указывать значение по умолчанию

In [93]:
def predefined(a, b, c = 3):
  print(a, b, c)

predefined(1,2)
predefined(3,4,5)

1 2 3
3 4 5


Можно, очевидно, делать это и для init'ов у классов

In [95]:
class Test():
  def __init__(self, path, a=100, b=10):
    self.path = path
    self.a = a
    self.b = b

print(Test("/test").__dict__)
print(Test("/test", 8).__dict__)
print(Test("/test", b=8).__dict__)

{'path': '/test', 'a': 100, 'b': 10}
{'path': '/test', 'a': 8, 'b': 10}
{'path': '/test', 'a': 100, 'b': 8}


Но они обязательно идут после аргументов без значения по умолчанию

In [96]:
def predefined(a = 1, b, c = 3):
  print(a, b, c)

predefined(1,2)
predefined(3,4,5)

SyntaxError: parameter without a default follows parameter with a default (1075962572.py, line 1)

## Работа с исключениями

Иногда в рамках выполнения кода мы можем хотеть предусмотреть исключительную ситуацию.

In [97]:
def divider(a, b):
  if b == 0:
    return None, "b == 0"
  return a/b, None

In [98]:
res, err = divider(1,2)

In [99]:
print(res)
print(err)

0.5
None


In [100]:
divider(10, 0)

(None, 'b == 0')

Но обычно мы хотим написать простую функцию

In [101]:
def divider(a, b):
  return a / b

И что-то может пойти не так

In [102]:
print(divider(1, 0))

ZeroDivisionError: division by zero

Для того, чтобы уберечь себя, мы можем использовать конструкцию try-except:
- try - что пытаемся сделать
- except - какие исключения пытаемся ловить и что делаем, если их поймаем
- else - что делаем, если не словили ошибку
- finally - что делаем при любом исходе

In [103]:
def main_func(a, b):
  # ....
  try:
    res = divider(a, b)
  except ZeroDivisionError:
    res = 0
  return res

In [104]:
main_func(1,0)

0

In [105]:
def safe_divider(a, b):
  try:
    res = a / b
  except ZeroDivisionError:
    print("unsafe division!")
    res = 0
  else:
    print("safe division!")
  finally:
    res = res ** 2
  return res

In [106]:
print(safe_divider(1,1))

safe division!
1.0


In [107]:
print(safe_divider(1,0))

unsafe division!
0


Исключения можно возвращать и самим

In [108]:
def exceptional_func():
  raise ValueError("test")

In [109]:
exceptional_func()

ValueError: test

In [110]:
def exceptional_func():
  return ValueError("test")

In [111]:
exceptional_func()

ValueError('test')

Исключения можно создавать и самим

In [113]:
class MyException(Exception):
  pass

def my_exceptional_func():
  raise MyException("my message")

In [114]:
my_exceptional_func()

MyException: my message

In [115]:
MyException.mro()

[__main__.MyException, Exception, BaseException, object]

In [116]:
def safe_divider(a, b):
  try:
    res = a / b
  except BaseException:
    print("unsafe division!")
    res = 0
  else:
    print("safe division!")
  finally:
    res = res ** 2
  return res

In [117]:
print(safe_divider(1,0))

unsafe division!
0


Except'ов может быть и несколько (и очень даже желательно)

In [ ]:
try:
  test_func()
except ValueError:
  print(1)
except KeyError:
  print(2)
except (TypeError, NameError):
  print(3)

In [125]:
def test(val):
  return 100 / val

def handler(val):
  val *= 2
  try:
    res = test(val)
  except ZeroDivisionError:
    res = -1
  print(res)

def main(val):
  try:
    handler(val)
  except ZeroDivisionError:
    print(":c")

In [126]:
main(0)

-1


## Декораторы

Есть 3 функции:
- декорирующая
- декорируемая
- та, которой декорируют

In [127]:
def calc(a, b): # декорируемая
  return a + b

calc(1,2)

3

In [130]:
def trace(func): # декорирующая (вызывается один раз)
  print("trace dec")
  def inner(*args, **kwargs): # та, которой декорируют (вызывается каждый раз)
    print("calling function:", func.__name__, ", parameters", args, kwargs)
    return func(*args, **kwargs)
  return inner

@trace # декорирующая (вызывается один раз)
def calc(a, b): # декорируемая
  return a + b

trace dec


In [131]:
calc(1,2)

calling function: calc , parameters (1, 2) {}


3

Декораторов может быть и много!  
При этом применяются они снизу вверх, а вызываются сверху вниз

In [132]:
def first(func):
  print("i'm the first")
  def inner(*args, **kwargs):
    print("first decoration")
    return func(*args, **kwargs)
  return inner

def second(func):
  print("i'm the second")
  def inner(*args, **kwargs):
    print("second decoration")
    return func(*args, **kwargs)
  return inner

@second
@first
def last():
  print("i'm the last")

i'm the first
i'm the second


In [133]:
last()

second decoration
first decoration
i'm the last


In [ ]:
second(first(last))

## Dataclasses

Делаем самый простой класс

In [134]:
class Author:
  def __init__(self, name, birthday, country="Russia"):
    self.name = name
    self.birthday = birthday

In [135]:
a = Author("Ivan", "01.01.1990")
print(a)

Но можно и не писать это все самим

In [136]:
from dataclasses import dataclass

@dataclass
class Author:
    name: str
    birthday: str

In [137]:
a = Author("Ivan", "01.01.1990")
print(a)

Author(name='Ivan', birthday='01.01.1990')


In [138]:
a.name = "Mike"
print(a)

Author(name='Mike', birthday='01.01.1990')


Есть и значения по умолчанию

In [139]:
@dataclass
class Author:
    name: str = "Ivan"
    birthday: str = "01.01.1990"

In [140]:
a = Author()
print(a)

Author(name='Ivan', birthday='01.01.1990')


А если мы хотим иметь иммутабельные экзмепляры?

In [141]:
@dataclass(frozen=True)
class Author:
    name: str
    birthday: str

In [142]:
a = Author("Ivan", "01.01.1990")
print(a)

Author(name='Ivan', birthday='01.01.1990')


In [143]:
a.name = "Mike"
print(a)

FrozenInstanceError: cannot assign to field 'name'

Интересно, что за нас написан не только init

In [144]:
class Author:
  def __init__(self, name, birthday):
    self.name = name
    self.birthday = birthday

In [145]:
Author("Ivan", "01.01.2000") == Author("Ivan", "01.01.2000")

False

In [146]:
@dataclass
class Author:
    name: str
    birthday: str

In [147]:
Author("Ivan", "01.01.2000") == Author("Ivan", "01.01.2000")

True

## Slots

Возможно, что мы хотим сделать ограничение на класс так, чтоб его экземплярам нельзя было добавить доп поля

In [148]:
class Test():
  def __init__(self):
    self.a = 1
    self.b = 2

t = Test()
t.c = 3

In [149]:
t.__dict__

{'a': 1, 'b': 2, 'c': 3}

In [150]:
class Test():
  __slots__ = ("a", "b")
  def __init__(self):
    self.a = 1
    self.b = 2

t = Test()
t.a = 10

In [151]:
t.c = 10

AttributeError: 'Test' object has no attribute 'c' and no __dict__ for setting new attributes

При наследовании слоты пропадают

In [157]:
class Test():
    __slots__ = ("a", "b")

class ChildTest(Test):
    pass

t = Test()
print(t.__slots__)
c = ChildTest()
print(ChildTest().__slots__)
c.c = 2
print(c.c)

('a', 'b')
('a', 'b')
2


Но можно и поправить эту проблему

In [158]:
class Test():
    __slots__ = ("a", "b")

class ChildTest(Test):
    __slots__ = ("c",)

c = ChildTest()
c.a = 1
c.c = 2
c.d = 5

AttributeError: 'ChildTest' object has no attribute 'd' and no __dict__ for setting new attributes

Однако с множественным наследованием так не выйдет

In [159]:
class BaseA(object):
    __slots__ = ('a',)

class BaseB(object):
    __slots__ = ('b',)

In [160]:
class Child(BaseA, BaseB):
    __slots__ = ('d',)

TypeError: multiple bases have instance lay-out conflict

## Перегрузка

С перегрузкой мы уже сталкивались ранее

In [161]:
class InsensitiveString():
  def __init__(self, string):
    self.string = string

s1 = InsensitiveString("abc")
s2 = InsensitiveString("ABC")
print(s1 == s2)
print(s1 != s2)

False
True


In [162]:
class InsensitiveString():
  def __init__(self, string):
    self.string = string

  def __add__(self, other):
    return self.string + other

In [163]:
InsensitiveString("abc") + "def"

'abcdef'

In [164]:
"abc" + InsensitiveString("def")

TypeError: can only concatenate str (not "InsensitiveString") to str

In [165]:
class InsensitiveString():
  def __init__(self, string):
    self.string = string

  def __add__(self, other):
    return self.string + other

  # нужно задавать, потому что Python сначала пытается сделать self.__add__(other), если не получается, то other.__radd__(self)
  def __radd__(self, other):
    return other + self.string

In [166]:
print(InsensitiveString("abc") + "def")
print("def" + InsensitiveString("abc"))

abcdef
defabc


In [ ]:
dir(InsensitiveString("abc"))

In [172]:
class InsensitiveString():
  def __init__(self, string):
    self.string = string

  def __eq__(self, other):
    return self.string.lower() == other.string.lower()

  def __ne__(self, other):
    return self.string.lower() != other.string.lower()

  def __add__(self, other):
    return self.string + other.string

  def __radd__(self, other):
    return other.string + self.string

  def __str__(self):
    return self.string.lower()

  def __repr__(self):
    return f"Insensitive string that has {self.string} value"

In [173]:
s1 = InsensitiveString("abc")
s2 = InsensitiveString("ABc")
print(s1 == s2)
print(s1 != s2)
print(s1 + s2)

True
False
abcABc


In [174]:
print(s1)
print(str(s1))
print(repr(s1))

abc
abc
Insensitive string that has abc value


У операторов сравнения есть интересная особенность

In [180]:
class Test:
  def __init__(self, val):
    self.val = val

  def __eq__(self, other):
    print("__eq__")
    return self.val == other.val

In [181]:
Test(1) == Test(1)

__eq__


True

In [182]:
Test(1) != Test(1)

__eq__


False

Как видим, хоть оператор и иной, а метод вызвался тот же; определим тогда и \_\_ne__

In [183]:
class Test:
  def __init__(self, val):
    self.val = val

  def __eq__(self, other):
    print("__eq__")
    return self.val == other.val

  def __ne__(self, other):
    print("__ne__")
    return self.val != other.val

In [184]:
Test(1) == Test(2)

__eq__


False

In [185]:
Test(1) != Test(2)

__ne__


True

Ожидаемо, и другие операторы сравнения ведут себя схожим образом

In [186]:
class Test:
  def __init__(self, val):
    self.val = val

  def __eq__(self, other):
    print("__eq__")
    return self.val == other.val

  def __ne__(self, other):
    print("__ne__")
    return self.val != other.val

  def __lt__(self, other):
    print("__lt__")
    return self.val < other.val

In [187]:
Test(1) < Test(2)

__lt__


True

In [188]:
Test(1) > Test(2)

__lt__


False

In [189]:
class Test:
  def __init__(self, val):
    self.val = val

  def __eq__(self, other):
    print("__eq__")
    return self.val == other.val

  def __ne__(self, other):
    print("__ne__")
    return self.val != other.val

  def __lt__(self, other):
    print("__lt__")
    return self.val < other.val

  def __gt__(self, other):
    print("__gt__")
    return self.val > other.val

In [190]:
Test(1) < Test(2)

__lt__


True

In [191]:
Test(1) > Test(2)

__gt__


False

Некоторые методы вызываются от нас очень неявно, к примеру, поведение при print(my_object) - ожидаем, что нам нужен \_\_str__, но питон и тут хитрит

In [193]:
class Author():
  def __init__(self, name, birth):
    self.name = name
    self.birth = birth

In [194]:
a = Author("Ivan", "01.01.2000")

In [195]:
print(a)

Объявим только \_\_repr__

In [196]:
class Author():
  def __init__(self, name, birth):
    self.name = name
    self.birth = birth

  def __repr__(self):
    return f"author, name: {self.name}, birthdate: {self.birth}"

a = Author("Ivan", "01.01.2000")
print(a)
print(str(a))
print(repr(a))
print(f"this is {a}")

author, name: Ivan, birthdate: 01.01.2000
author, name: Ivan, birthdate: 01.01.2000
author, name: Ivan, birthdate: 01.01.2000
this is author, name: Ivan, birthdate: 01.01.2000


In [197]:
class Author():
  def __init__(self, name, birth):
    self.name = name
    self.birth = birth

  def __repr__(self):
    return f"author, name: {self.name}, birthdate: {self.birth}"

  def __str__(self):
    return f"author named {self.name}"

a = Author("Ivan", "01.01.2000")
print(a)
print(str(a))
print(repr(a))
print(f"this is {a}")

author named Ivan
author named Ivan
author, name: Ivan, birthdate: 01.01.2000
this is author named Ivan


## Сеттеры и Геттеры

In [198]:
class MyClass():
  def __init__(self):
    self.value = 0

Конечно, мы можем напрямую работать с полем __value__

In [199]:
c = MyClass()
c.value += 1
print(c.value)

1


Однако в таком случае мы можем случайно изменять значения полей, что может быть критичным. Более того, мы можем хотеть скрыть внутренности класса.

In [200]:
class MyProtectedClass():
  def __init__(self):
    self.__value = 0

  def value(self):
    return self.__value

  def set_value(self, new_value):
    self.__value = new_value

In [201]:
c = MyProtectedClass()
c.set_value(5)
c.value()

5

## Контекст

Обычно контекст используют при работе с файлами:

In [202]:
f = open("test.txt", "w")
f.write("test line")
f.close()

In [ ]:
with open("test2.txt", "w") as f:
  f.write("test line")

Но на самом деле мы можем написать свой менеджер контекста:

In [203]:
class Manager():
    def __enter__(self):
        print("enter")

    def __exit__(self, type, value, traceback):
        print("exit")

with Manager() as m:
  print("managing")

enter
managing
exit


## Pattern matching (Python 3.10+)

In [204]:
def do_stuff_v1(value):
  if value == "a":
    print(1)
  elif value == "b":
    print(2)
  elif value == "c":
    print(3)
  else:
    print(0)

In [205]:
do_stuff_v1("a")
do_stuff_v1("asd")

1
0


In [206]:
def do_stuff_v2(value):
  match value:
    case "a":
      print(1)
    case "b":
      print(2)
    case "c":
      print(3)
    case _:
      print(0)

In [207]:
do_stuff_v2("a")
do_stuff_v2("asd")

1
0


## Кэширование

Часто нам необходимо долго обрабатывать данные, при этом входные данные могут повторяться. Для того, чтобы сэкономить себе время, мы можем сделать себе кэш

In [208]:
import time

In [209]:
def compute_output(input):
  time.sleep(2)
  return input * 10

def compute(input):
  output = compute_output(input)
  return output

In [210]:
%%time
compute("a")

CPU times: user 1.06 ms, sys: 0 ns, total: 1.06 ms
Wall time: 2 s


'aaaaaaaaaa'

In [211]:
%%time
compute("a")

CPU times: user 947 µs, sys: 0 ns, total: 947 µs
Wall time: 2 s


'aaaaaaaaaa'

Добавим же к методу работу с кэшом

In [212]:
cache = {} # ключ - input, значение - output

def compute(input):
  if input in cache.keys():
    return cache[input]
  output = compute_output(input)
  cache[input] = output
  return output

In [213]:
%%time
compute("b")

CPU times: user 659 µs, sys: 0 ns, total: 659 µs
Wall time: 2 s


'bbbbbbbbbb'

In [214]:
%%time
compute("b")

CPU times: user 6 µs, sys: 0 ns, total: 6 µs
Wall time: 8.11 µs


'bbbbbbbbbb'

## Указание типов

Мы можем писать, какие мы ожидаем увидеть типы у аргументов и результатов

In [215]:
import math

In [216]:
def power(value_1, value_2):
  return math.pow(value_1, value_2)

In [ ]:
power()

In [217]:
def power(value_1: int, value_2: int) -> int:
  return int(math.pow(value_1, value_2))

In [218]:
print(power(2, 3))

8


Но это все просто комментарии..

In [219]:
print(power("s", 3))

TypeError: must be real number, not str

## Нейминг

Именование переменных, функций, классов - важная вещь. По идее, одного взгляда на них должно быть достаточно доя того, чтобы понять, для чего они существуют

In [220]:
def func(a, b):
  return [x * y for x in a for y in b]

def array_zipper(source, source_2):
  return [x * y for x in source for y in source_2]

In [ ]:
def copy(a, b):
  for item in a:
    b.append(item)

In [ ]:
def copy(source, dest):
  for item in source:
    dest.append(item)

In [ ]:
def copy(dest: list, source: list):
  for item in source:
    dest.append(item)

In [221]:
a = [1,2,3]
b = [2,4]
print(func(a, b))

[2, 4, 4, 8, 6, 12]


In [222]:
print(array_zipper(a, b))

[2, 4, 4, 8, 6, 12]


In [ ]:
for range_counter in range(0, 10):
  range_counter += 1
  pass

При этом очевидно, что все очень сильно зависит от того, какое время жизни у переменной - если переменная нужна лишь на одной строке, нет ничего зазорного в том, чтобы назвать ее __a__ или __x__

## Тесты

https://realpython.com/python-testing/#unit-tests-vs-integration-tests

Грубо говоря, тесты можно поделить на:

- интеграционные тесты
- юнит тесты

### Assert

In [223]:
assert 1==1, "test 1"

In [224]:
assert 1==2, "test 2"

AssertionError: test 2

In [225]:
assert 1==3

AssertionError: 

### Unittest

In [226]:
import unittest

In [227]:
class TestNotebook(unittest.TestCase):
  def test_1(self):
    self.assertEqual(1, 1)

  def test_2(self):
    self.assertEqual(1, 2)

In [228]:
unittest.main(argv=[''], verbosity=2, exit=False)

test_1 (__main__.TestNotebook.test_1) ... ok
test_2 (__main__.TestNotebook.test_2) ... FAIL

FAIL: test_2 (__main__.TestNotebook.test_2)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_670/1506563387.py", line 6, in test_2
    self.assertEqual(1, 2)
    ~~~~~~~~~~~~~~~~^^^^^^
AssertionError: 1 != 2

----------------------------------------------------------------------
Ran 2 tests in 0.005s

FAILED (failures=1)


In [229]:
class TestNotebook(unittest.TestCase):
  def test_eq(self):
    self.assertEqual(1, 1) # ==

  def test_not_eq(self):
    self.assertNotEqual(1, 2) # !=

  def test_true(self):
    a = True
    self.assertTrue(a) # == True

  def test_false(self):
    a = False
    self.assertFalse(a) # == False

  def test_is(self):
    a = 123
    b = 123
    self.assertIs(a, b) # is

  def test_is_not(self):
    a = {}
    b = {}
    self.assertIsNot(a, b) # is not

  def test_is_none(self):
    a = None
    self.assertIsNone(a)  # is None

  def test_is_not_none(self):
    a = "None"
    self.assertIsNotNone(a)

  def test_in(self):
    a = 1
    b = [1,2]
    self.assertIn(a, b)

  def test_not_in(self):
    a = 3
    b = [1,2]
    self.assertNotIn(a, b)

  def test_is_instance(self):
    a = ""
    self.assertIsInstance(a, str)

  def test_not_is_instance(self):
    a = 1
    self.assertNotIsInstance(a, str)

In [230]:
unittest.main(argv=[''], verbosity=2, exit=False)

test_eq (__main__.TestNotebook.test_eq) ... ok
test_false (__main__.TestNotebook.test_false) ... ok
test_in (__main__.TestNotebook.test_in) ... ok
test_is (__main__.TestNotebook.test_is) ... ok
test_is_instance (__main__.TestNotebook.test_is_instance) ... ok
test_is_none (__main__.TestNotebook.test_is_none) ... ok
test_is_not (__main__.TestNotebook.test_is_not) ... ok
test_is_not_none (__main__.TestNotebook.test_is_not_none) ... ok
test_not_eq (__main__.TestNotebook.test_not_eq) ... ok
test_not_in (__main__.TestNotebook.test_not_in) ... ok
test_not_is_instance (__main__.TestNotebook.test_not_is_instance) ... ok
test_true (__main__.TestNotebook.test_true) ... ok

----------------------------------------------------------------------
Ran 12 tests in 0.019s

OK


До и после каждого теста можно что-то делать

In [231]:
class TestNotebook(unittest.TestCase):
  def setUp(self):
    print("setting up")

  def test_1(self):
    self.assertEqual(1, 1)

  def test_2(self):
    self.assertEqual(1, 2)

  def tearDown(self):
    print("tearing down")

In [232]:
unittest.main(argv=[''], verbosity=2, exit=False)

test_1 (__main__.TestNotebook.test_1) ... ok
test_2 (__main__.TestNotebook.test_2) ... FAIL

FAIL: test_2 (__main__.TestNotebook.test_2)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_670/2104376223.py", line 9, in test_2
    self.assertEqual(1, 2)
    ~~~~~~~~~~~~~~~~^^^^^^
AssertionError: 1 != 2

----------------------------------------------------------------------
Ran 2 tests in 0.004s

FAILED (failures=1)


setting up
tearing down
setting up
tearing down


In [233]:
class TestNotebook(unittest.TestCase):
  def setUp(self):
    print("setting up")

  def test_1(self):
    self.assertEqual(1, 1)

  def test_2(self):
    self.assertEqual(1, 2)

  def tearDown(self):
    print("tearing down")

  @classmethod
  def setUpClass(cls):
    print("before all tests")

  @classmethod
  def tearDownClass(cls):
    print("after all tests")

In [234]:
unittest.main(argv=[''], verbosity=2, exit=False)

test_1 (__main__.TestNotebook.test_1) ... ok
test_2 (__main__.TestNotebook.test_2) ... FAIL

FAIL: test_2 (__main__.TestNotebook.test_2)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_670/2772796241.py", line 9, in test_2
    self.assertEqual(1, 2)
    ~~~~~~~~~~~~~~~~^^^^^^
AssertionError: 1 != 2

----------------------------------------------------------------------
Ran 2 tests in 0.007s

FAILED (failures=1)


before all tests
setting up
tearing down
setting up
tearing down
after all tests


Тесты можно запускать и не все

In [237]:
import sys
sys.platform

'linux'

In [238]:
class TestNotebook(unittest.TestCase):
  @unittest.skip("skip")
  def test_skip(self):
      self.assertEqual(1, 1)

  @unittest.skipIf(not sys.platform.startswith("win"), "requires Windows")
  def test_skip_not_win(self):
      self.assertEqual(1, 1)

  def test_not_skip(self):
    self.assertEqual(1, 1)

In [239]:
unittest.main(argv=[''], verbosity=2, exit=False)

test_not_skip (__main__.TestNotebook.test_not_skip) ... ok
test_skip (__main__.TestNotebook.test_skip) ... skipped 'skip'
test_skip_not_win (__main__.TestNotebook.test_skip_not_win) ... skipped 'requires Windows'

----------------------------------------------------------------------
Ran 3 tests in 0.005s

OK (skipped=2)


### Selenium

In [240]:
!pip install google-colab-selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 93.5 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [1]:
import google_colab_selenium as gs

driver = gs.Chrome()
driver.quit()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import sys
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service

In [3]:
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless') # ensure GUI is off
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--verbose')
chrome_options.add_argument('--log-path=/content/test.log')

In [4]:
wd = gs.Chrome(options=chrome_options)

<IPython.core.display.Javascript object>

In [5]:
wd.get("http://www.python.org")
assert "Python" in wd.title
elem = wd.find_element(By.NAME, "q")
elem.clear()
elem.send_keys("pycon")
elem.send_keys(Keys.RETURN)
assert "No results found." not in wd.page_source
wd.quit()

In [ ]:
wd = gs.Chrome(options=chrome_options)
wd.get("http://www.python.org")
assert "Python" in wd.title
elem = wd.find_element(By.NAME, "q")
elem.clear()
elem.send_keys("pycon")
elem.send_keys(Keys.RETURN)
assert "No results found." in wd.page_source
wd.quit()

In [ ]:
wd.quit()

Чтобы всегда корректно работать с драйвером, лучше делать непосредственно тест + setUp и tearDown

In [7]:
import unittest

In [8]:
class TestNotebook(unittest.TestCase):
  def setUp(self):
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    self.wd = gs.Chrome(options=chrome_options)

  def test_get_python(self):
    self.wd.get('http://www.python.org')
    self.assertIn("Python", self.wd.title)

    button = self.wd.find_element(By.CLASS_NAME, "donate-button")
    self.assertIsNotNone(button)

    button.click()

    print(self.wd.current_url)
    self.assertEqual(self.wd.current_url, "https://psfmember.org/civicrm/contribute/transact/?reset=1&id=2")

  def tearDown(self):
    self.wd.quit()

In [9]:
unittest.main(argv=[''], verbosity=2, exit=False)

test_get_python (__main__.TestNotebook.test_get_python) ... 

<IPython.core.display.Javascript object>

FAIL

FAIL: test_get_python (__main__.TestNotebook.test_get_python)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_37379/3157643271.py", line 19, in test_get_python
    self.assertEqual(self.wd.current_url, "https://psfmember.org/civicrm/contribute/transact/?reset=1&id=2")
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 'https://www.python.org/psf/donations/' != 'https://psfmember.org/civicrm/contribute/transact/?reset=1&id=2'
- https://www.python.org/psf/donations/
+ https://psfmember.org/civicrm/contribute/transact/?reset=1&id=2


----------------------------------------------------------------------
Ran 1 test in 1.953s

FAILED (failures=1)


https://www.python.org/psf/donations/


# Некоторые из частых ошибок

## Работа с массивом, а не с его копией

In [10]:
aa = [1,2,3,4,5]
for i in range(0, len(aa)):
  print(i, len(aa))
  a = aa[i]
  if a == 3 or a == 4:
    aa.pop(i)
print(aa)

0 5
1 5
2 5
3 4
4 4


IndexError: list index out of range

In [11]:
aa = [1,2,3,4,5]
for a in aa:
  if a == 3 or a == 4:
    aa.remove(a)
print(aa)

[1, 2, 4, 5]


In [12]:
aa = [1,2,3,4,5]
for a in aa[:]:
  if a == 3 or a == 4:
    aa.remove(a)
print(aa)

[1, 2, 5]


## Работа с диктом, а не с его копией

In [13]:
aa = {1: 1, 2: 2, 3: 3, 4: 4, 5: 5}
for k, v in aa.items():
  del aa[k]
print(aa)

RuntimeError: dictionary changed size during iteration

In [15]:
aa = {1: 1, 2: 2, 3: 3, 4: 4, 5: 5}
for k in aa.keys():
  del aa[k]
print(aa)

RuntimeError: dictionary changed size during iteration

In [14]:
aa = {1: 1, 2: 2, 3: 3, 4: 4, 5: 5}
for k in aa.copy():
  del aa[k]
print(aa)

{}


##Значение по умолчанию, являющееся изменяемым объектом

In [16]:
def test(a=10):
  return a

print(test())
print(test(1))

10
1


In [17]:
def test_list(a=[]):
  a.append("a")
  return a

print(test_list())

['a']


In [18]:
print(test_list())

['a', 'a']


In [19]:
def test_list(a=None):
  if not a:
    a = []
  a.append("a")
  return a

print(test_list())

['a']


In [20]:
print(test_list())

['a']


In [21]:
def test_dict(key, a={}):
  a[key] = 0
  return a

print(test_dict("a"))
print(test_dict("b"))

{'a': 0}
{'a': 0, 'b': 0}


##Проверка на True/False

In [22]:
def test(val):
  if val == True:
    print("eq True")

  if val != True:
    print("ne True")

  if val == False:
    print("eq False")

  if val != False:
    print("ne False")

In [23]:
test(True)

eq True
ne False


In [24]:
test(False)

ne True
eq False


In [25]:
test(None)

ne True
ne False


In [26]:
def test(val):
  if val == True:
    print("eq True")

  if val != True:
    print("ne True")

  if val == False:
    print("eq False")

  if val != False:
    print("ne False")

  if val:
    print("if")

  if not val:
    print("if not")

  if val is True:
    print("is True")

  if val is False:
    print("is False")

  if val is None:
    print("is None")

In [27]:
test(True)

eq True
ne False
if
is True


In [28]:
test(False)

ne True
eq False
if not
is False


In [29]:
test(None)

ne True
ne False
if not
is None


In [30]:
test(1)

eq True
ne False
if


# Мини домашка

## Задание 1

Перепишите функцию divisible() так, чтобы вместо вложенных циклов использовался list comprehension

In [ ]:
def divisible():
  res = []
  for n in range(1, 100):
    for x in range(2, 10):
      if n % x == 0:
        res.append(n)

  return res

In [ ]:
def divisible_comprehension():
  # тут ваш код
  return []

assert divisible() == divisible_comprehension()

## Задание 2

Используйте форматирование строк и переопределение метода \_\_repr__, получив ожидаемый результат

In [ ]:
class MyList():
  def __init__(self, data):
    self.data = data

  def __repr__(self):
    # ваш код
    template = ""
    return template.format()

In [ ]:
data = [4, 30, 2017, 2, 27]
expected = 'initial order is 4 30 2017 2 27, new order is 2 27 2017 4 30'

assert repr(MyList(data)) == expected

## Задание 3

Реализуйте метод concat() таким образом, чтобы функция:
- принимала произвольные неименованные и именованные аргументы
- неименованные аргументы просто добавились в строку с результатом
- именованные аргументы были отсортированы по ключу и добавлены в строку с результатом в формате key=value

In [ ]:
def concat(*args, **kwargs):
  res = []

  # работа с args

  # работа с kwargs

  return ','.join(res)

In [ ]:
assert concat(5,3,1) == '5,3,1'
assert concat(1,2,'4',False,k2=1,k1='test') == '1,2,4,False,k1=test,k2=1'
assert concat(k1='test',k2='test2') == 'k1=test,k2=test2'

## Задание 4

Для метода деления реализуйте:
- декоратор для обработки исключения, которое возникает при делении на 0, возвращая 0
- декоратор с кэшом, чтобы значение бралось из него, если уже для этих значений деление производили раньше

In [ ]:
import time
import unittest

In [ ]:
def cache(func):

  def inner(*args, **kwargs):

  return inner

def exception_catcher(func):

  def inner(*args, **kwargs):

  return inner

@cache
@exception_catcher
def divider(a, b):
  time.sleep(1)
  return a/b

In [ ]:
class MyTestCase(unittest.TestCase):
  def testDiv0(self):
    self.assertEqual(0, divider(1, 0))

  def testDivStr(self):
    with self.assertRaises(TypeError):
      divider(1, "a")

  def testOk(self):
    self.assertEqual(2, divider(4, 2))

  def testNoCache(self):
    start = time.time()
    res = divider(35, 7)
    self.assertTrue(time.time() - start > 1)
    self.assertEqual(5, res)

  def testCache(self):
    start = time.time()
    res = divider(40, 8)
    self.assertTrue(time.time() - start < 1)
    self.assertEqual(5, res)

  @classmethod
  def setUpClass(cls):
    # fill cache
    divider(40, 8)

  @classmethod
  def tearDownClass(cls):
    pass

In [ ]:
unittest.main(argv=[''], verbosity=2, exit=False)